# 08 — Inferência em novos clientes (K-Means)

Objetivos:
- Carregar a base `data/new/Customer Sucess_novos_clientes.xlsx`
- Aplicar o pipeline já treinado (sem re-treinar)
- Listar nomes por cluster no formato solicitado
- Exportar `reports/novos_clientes_clusterizados.xlsx` para auditoria


In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path('..').resolve()))

import numpy as np
import pandas as pd
import joblib

PROJECT_ROOT = Path('..').resolve()
REPORTS_DIR = PROJECT_ROOT / 'reports'
DATA_NEW_PATH = PROJECT_ROOT / 'data' / 'new' / 'Customer Sucess_novos_clientes.xlsx'

PIPELINE_PATH = REPORTS_DIR / 'kmeans_pipeline.joblib'
PIPELINE_FALLBACK_PATH = REPORTS_DIR / 'models' / 'kmeans_pipeline.joblib'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('Base novos clientes:', DATA_NEW_PATH)
print('Pipeline (primário):', PIPELINE_PATH)
print('Pipeline (fallback):', PIPELINE_FALLBACK_PATH)

if not DATA_NEW_PATH.exists():
    raise FileNotFoundError(
        f'Arquivo de novos clientes não encontrado em: {DATA_NEW_PATH}. '\
        'Garanta que o arquivo foi movido para data/new/ conforme instruções.'
    )

if not PIPELINE_PATH.exists() and not PIPELINE_FALLBACK_PATH.exists():
    raise FileNotFoundError(
        'Pipeline não encontrado. Esperado em: '\
        f'{PIPELINE_PATH} (ou {PIPELINE_FALLBACK_PATH}).'
    )


In [ ]:
xl = pd.ExcelFile(DATA_NEW_PATH)
sheet = xl.sheet_names[0]
print('Aba detectada:', sheet)

df_raw = pd.read_excel(DATA_NEW_PATH, sheet_name=sheet)
print('Shape:', df_raw.shape)
df_raw.head(3)


In [ ]:
def find_name_col(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    by_lower = {str(c).strip().lower(): c for c in cols}

    preferred = ['cliente', 'nome', 'aluno', 'nome do aluno', 'nome_aluno', 'student', 'customer']
    for k in preferred:
        if k in by_lower:
            return str(by_lower[k])

    for c in cols:
        cl = str(c).strip().lower()
        if 'cliente' in cl:
            return str(c)
        if 'nome' in cl:
            return str(c)
        if 'aluno' in cl:
            return str(c)

    raise ValueError(
        'Não foi possível identificar a coluna de nome do cliente. '
        "Tente usar uma coluna chamada 'CLIENTE' (ou 'cliente').\n"
        f'Colunas disponíveis: {cols}'
    )


NAME_COL_RAW = find_name_col(df_raw)
print('Coluna de nome detectada:', NAME_COL_RAW)


In [ ]:
from src.preprocessing import clean_base, build_cliente_atual
from src.features import build_modeling_dataframe

df_clean, meta = clean_base(df_raw)
print('Colunas padronizadas (snake_case). Exemplo:', list(df_clean.columns)[:12])
print('Flags criadas:', meta.created_flags)

df_cliente_atual = build_cliente_atual(df_clean)
print('Shape cliente-atual:', df_cliente_atual.shape)

X, _spec = build_modeling_dataframe(df_cliente_atual)
print('Shape features (X):', X.shape)
X.head(3)


In [ ]:
pipeline_path_used = PIPELINE_PATH if PIPELINE_PATH.exists() else PIPELINE_FALLBACK_PATH
pipeline = joblib.load(pipeline_path_used)
print('Pipeline carregado de:', pipeline_path_used)

pre = pipeline.named_steps.get('preprocess')
if pre is None:
    raise ValueError('Pipeline não contém a etapa preprocess esperada.')

num_cols = list(pre.transformers_[0][2])
cat_cols = list(pre.transformers_[1][2])

for c in num_cols:
    if c not in X.columns:
        X[c] = 0.0
for c in cat_cols:
    if c not in X.columns:
        X[c] = 'desconhecido'

X = X[num_cols + cat_cols]

cluster_id = pipeline.predict(X).astype(int)

cluster_map = {
    4: '🟢 Champions',
    0: '🟡 Potenciais',
    1: '🟠 Avulsos Engajados',
    2: '🔴 Zumbis',
    3: '⚪ Churn Iminente',
    5: '🟣 Novos',
}

nome_cluster = pd.Series(cluster_id).map(cluster_map)
if nome_cluster.isna().any():
    missing = sorted(set(pd.Series(cluster_id)[nome_cluster.isna()].tolist()))
    raise ValueError(f'Foram encontrados clusters não mapeados: {missing}')


In [ ]:
out = df_cliente_atual.copy()
out['cluster_id'] = cluster_id
out['nome_cluster'] = nome_cluster

name_series = out.get('cliente')
if name_series is None:
    raise ValueError('Coluna `cliente` não encontrada após limpeza. Verifique a base de novos clientes.')

order = [4, 0, 1, 2, 3, 5]
headers = {
    4: '🟢 Champions - Cluster 4:',
    0: '🟡 Potenciais - Cluster 0:',
    1: '🟠 Avulsos Engajados - Cluster 1:',
    2: '🔴 Zumbis - Cluster 2:',
    3: '⚪ Churn Iminente - Cluster 3:',
    5: '🟣 Novos - Cluster 5:',
}

for cid in order:
    print(headers[cid])
    nomes = name_series[out['cluster_id'] == cid].dropna().astype(str).tolist()
    if not nomes:
        print('— (nenhum cliente)')
    else:
        for n in nomes:
            print(n)
    print('')


In [ ]:
export_cols = [
    'cliente',
    'cluster_id',
    'nome_cluster',
    'log_acessos',
    'dias_sem_acessar',
    'recorrente',
    'parcelado',
    'nunca_acessou',
    'metodo_pagamento',
    'n_transacoes_cliente',
    'recencia_compra_dias',
]
export_cols = [c for c in export_cols if c in out.columns]

df_export = out[export_cols].copy()
if NAME_COL_RAW.upper() != 'CLIENTE':
    df_export = df_export.rename(columns={'cliente': NAME_COL_RAW})

out_path = REPORTS_DIR / 'novos_clientes_clusterizados.xlsx'
with pd.ExcelWriter(out_path, engine='xlsxwriter') as writer:
    df_export.to_excel(writer, sheet_name='novos_clientes', index=False)

print('Export salvo em:', out_path)
df_export.head(3)


In [ ]:
n_total = len(out)
dist = out['cluster_id'].value_counts().reindex([4, 0, 1, 2, 3, 5]).fillna(0).astype(int)
dist_pct = (dist / n_total * 100).round(2)
summary = pd.DataFrame({'cluster_id': dist.index, 'n': dist.values, 'pct': dist_pct.values})

print('Total de novos clientes processados:', n_total)
summary
